In [ ]:
import pandas as pd
import numpy as np

data = pd.read_csv('../../Excercise_data/L05/core_data.csv')
data

👉 Take the time to do a preliminary investigation of the features. Notice that you have a column names 'NGR total counts (cps)': this represents the natural gamma ray reading of the core, and there is a complete explanation available [here](http://www-odp.tamu.edu/publications/tnotes/tn26/CHAP5.PDF). You will also see columns named 'L*', 'a*', and 'b*': this refers to the reflectance data from the core, expressed in the CIELAB color space. You can learn much more about this by reading [this techincal note](http://www-odp.tamu.edu/publications/tnotes/tn26/CHAP7.PDF). Reading the technical notes is not essential to understanding this exercise, so consider this optional but good to have.

# Duplicates


In [ ]:
data = data.drop_duplicates()


# Splitting your data
Now split your dataset into a `train_set` (80%) and a `test_set` (20%). Use `42` as your `random_state`.

In [ ]:
from sklearn.model_selection import train_test_split

train_set, test_set = train_test_split(data, train_size=.8, random_state=42)

# Missing data
From now on, use <span style="color:red">only your `train_set`</span>, ***NOT*** your `data` or your `test_set`.
👇 Print out the percentage of missing values for all columns of the dataframe.

In [ ]:
100 * train_set.isnull().mean()

## `Lithology`

👇 Investigate the missing values in `Lithology`. Then, chose one of the following solutions:

1. Drop the column entirely
2. Impute the column median using Sklearn's `SimpleImputer`
3. Preserve the NaNs and replace by actual meaning

Make changes in `train_set` only; keep fitted transformers for later test transformation.

<details>
    <summary>💡 Hint</summary>
    ℹ️ <code>Lithology</code> has a lot of missing values. The description does not touch on what they represent. As such, it is better not to make any assumptions and to drop the column entirely.
</details>

In [ ]:
train_set = train_set.drop('Lithology', axis=1)

## `NGR total counts (cps)`
This is the response we want to predict in the feature-selection exercise. Exclude rows with missing response values; do not impute a target and treat it as an observed label. Separate `y_train` from the predictors before preprocessing.

<details><summary>💡 Hint</summary>
Use `dropna(subset=[...])`, then `pop` to remove the target from the feature dataframe while retaining its index.
</details>

In [ ]:
from sklearn.impute import SimpleImputer

train_set = train_set.dropna(subset=['NGR total counts (cps)']).copy()
y_train = train_set.pop('NGR total counts (cps)')

👇 When you are done, print out the percentage of missing values for the entire dataframe.

In [ ]:
100 * train_set.isnull().sum().sum() / train_set.size

⚠️ Be careful: not all missing values are represented `np.nans`, and python's `isnull()` only detects `np.nans` ⚠️

## `Reflectance L*, a* and b*`

👇 Investigate the missing values in `Reflectance L*, a*, b*`. Then, chose one of the following solutions:

1. Drop the column entirely
2. Impute the column mean using Sklearn's `SimpleImputer`
3. Preserve the NaNs and replace by actual meaning

Make changes in `train_set` only; keep fitted transformers for later test transformation.


<details>
    <summary>💡 Hint</summary>
ℹ️ <code>Reflectance L*, a*, b*</code> have few missing values that can be imputed by the mean value. You can do this with a single imputer.
</details>

In [ ]:
RefImputer = SimpleImputer(strategy='mean')
train_set[['Reflectance L*', 'Reflectance a*','Reflectance b*']]=RefImputer.fit_transform(train_set[['Reflectance L*','Reflectance a*','Reflectance b*']])

## `Type`

👇 Investigate the missing values in `Type`. Then, chose one of the following solutions:

1. Drop the column entirely
2. Impute the column median
3. Preserve the NaNs and replace by actual meaning

Make changes in `train_set` only; keep fitted transformers for later test transformation.


<details>
    <summary>💡 Hint</summary>
ℹ️ <code>Type</code> represents the type of the coring apparatus used. It is a string, not a number. Only a few values are missing, and you don't want to loose important information by dropping the rows: instead you could assume that the type can be replaced by the most frequent value without loosing much information. Check the <code>SimpleImputer</code> documentation to see how to do that. 
</details>

In [ ]:
typeImputer = SimpleImputer(strategy='most_frequent')
train_set[['Type']] = typeImputer.fit_transform(train_set[['Type']])

## `Depth CSF-A (m)`

👇 Investigate the missing values in `Depth CSF-A (m)`. Remember that this represents the depth of the sample below surface, a critical piece of information that cannot easily be supplemented by other data. With this in mind, chose one of the following solutions:

1. Drop the rows entirely
2. Impute the column median
3. Preserve the NaNs and replace by actual meaning

Make changes in `train_set` only; keep fitted transformers for later test transformation.


<details>
    <summary>💡 Hint</summary>
ℹ️ <code>Depth CSF-A (m)</code> is only missing in a few rows. Because we cannot supplement it easily and because a 'mean' depth for a sample has no real meaning, a good strategy here would be to drop the rows where this value is NaN. Look at the pandas documentation to see how to drop specific rows: think also of how you would find the index of the rows that need dropping.
</details>

In [ ]:
index = train_set[train_set['Depth CSF-A (m)'].isnull() == True].index

train_set.drop(index, inplace=True)
y_train = y_train.loc[train_set.index]

### Check to see if you still have missing values

In [ ]:
train_set.isnull().sum()

# Scaling
In order to investigate features and, we recommend that you plot a histogram and a box plot for each one of the features.

##  `Reflectance L*` 

👇 Investigate `Reflectance L*` for distribution and outliers. Then, choose the most appropriate scaling technique. Either:

1. Standard Scale
2. Robust Scale
3. MinMax Scale

Replace the original columns by the transformed values.

In [ ]:
import matplotlib.pyplot as plt

plt.hist(train_set['Reflectance L*']);

In [ ]:
train_set[['Reflectance L*']].boxplot();

<details><summary>💡 Hint</summary>
`StandardScaler` centres and scales using the training mean and standard deviation. It does not require or create a normal distribution. Retain the fitted scaler for future observations.
</details>

In [ ]:
from sklearn.preprocessing import StandardScaler

LScaler = StandardScaler()
train_set[['Reflectance L*']] = LScaler.fit_transform(train_set[['Reflectance L*']])

## `Refectance a*` & `Reflectance b*`

👇 Investigate `Reflectance a*` & `Reflectance b*`. Then, chose one of the following scaling techniques:

1. MinMax Scale
2. Standard Scale
3. Robust Scale

Replace the original columns by the transformed values.

<details>
    <summary>💡 Hint</summary>
ℹ️ <code>Reflectance a*</code> and <code>Reflectance b*</code> are both normally distributed but with some outliers: we could use the <code>RobustScaler()</code> here.
</details>

In [ ]:
import matplotlib.pyplot as plt

plt.hist(train_set['Reflectance a*']);

In [ ]:
train_set[['Reflectance a*']].boxplot()

In [ ]:
import matplotlib.pyplot as plt

plt.hist(train_set['Reflectance b*']);

In [ ]:
train_set[['Reflectance b*']].boxplot()

In [ ]:
from sklearn.preprocessing import RobustScaler

abScaler = RobustScaler()
train_set[['Reflectance a*','Reflectance b*']] = abScaler.fit_transform(train_set[['Reflectance a*','Reflectance b*']]) 

## `Depth CSF-A (m)`

👇 Investigate `Depth CSF-A (m)` for distribution and outliers. Then, choose the most appropriate scaling technique. Either:

1. Standard Scale
2. Robust Scale
3. MinMax Scale

Replace the original columns by the transformed values.

In [ ]:
import matplotlib.pyplot as plt

plt.hist(train_set['Depth CSF-A (m)']);

In [ ]:
train_set[['Depth CSF-A (m)']].boxplot();

<details><summary>💡 Hint</summary>
`MinMaxScaler` maps the observed training minimum and maximum to 0 and 1. New values outside that range can transform outside [0, 1]. The choice of scaler depends on the model and sensitivity to outliers, not a normality requirement.
</details>

In [ ]:
from sklearn.preprocessing import MinMaxScaler

depthScaler = MinMaxScaler()

train_set['Depth CSF-A (m)'] = depthScaler.fit_transform(train_set[['Depth CSF-A (m)']])

# Feature Engineering

## `Type`

👇 Investigate `Type` and chose one of the following encoding techniques accordingly:
- Ordinal encoding
- One-Hot encoding

Add the encoding to the dataframe as new colum(s), and remove the original column.


<details>
    <summary>💡 Hint</summary>
ℹ️ `Type` is a multicategorical feature that must be One hot encoded: there is no explicit ordinal value to the drill bit type.
</details>

In [ ]:
# Check what unique value of types exist:
train_set.Type.unique()

In [ ]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore').fit(train_set[['Type']])

In [ ]:
ohe.categories_

In [ ]:
# Derive labels from the fitted encoder; never assume the category order.
type_columns = ohe.categories_[0].tolist()
train_set[type_columns] = ohe.transform(train_set[['Type']])
train_set.drop('Type', axis=1, inplace=True)
train_set

# 🏁 Apply the fitted transformations
Transform the held-out observations with the **same fitted** imputers, scalers and encoder. Do not fit them again on test data. These prepared data are suitable for a fixed train/test evaluation; for cross-validation, place the learned steps inside a pipeline so each fold fits them separately.

In [ ]:
test_features = test_set.dropna(
    subset=['NGR total counts (cps)', 'Depth CSF-A (m)']).copy()
y_test = test_features.pop('NGR total counts (cps)')
test_features = test_features.drop(columns='Lithology')
reflectance = ['Reflectance L*', 'Reflectance a*', 'Reflectance b*']
test_features[reflectance] = RefImputer.transform(test_features[reflectance])
test_features[['Type']] = typeImputer.transform(test_features[['Type']])
test_features[['Reflectance L*']] = LScaler.transform(test_features[['Reflectance L*']])
test_features[['Reflectance a*', 'Reflectance b*']] = abScaler.transform(
    test_features[['Reflectance a*', 'Reflectance b*']])
test_features[['Depth CSF-A (m)']] = depthScaler.transform(test_features[['Depth CSF-A (m)']])
test_features[type_columns] = ohe.transform(test_features[['Type']])
test_features = test_features.drop(columns='Type')
test_features.head()